# Flood Disruption Analysis of Road Networks

This notebook implements a flood disruption framework for urban road networks. It proceeds through four sequential stages:

1. **Flood inundation attribution** — Samples flood water depth from raster mosaics (10 return periods) onto each network node and edge.
2. **Network disruption** — Removes nodes and edges inundated above a 0.3 m threshold and recalculates travel speeds using a depth–speed relationship.
3. **Exposure quantification** — Computes total exposed road length across a sensitivity range of depth thresholds for every return period.
4. **Disrupted network generation** — Produces and serialises final disrupted graphs for downstream accessibility or routing analysis.

Documentation Note: Portions of this documentation were drafted with the assistance of Claude (Anthropic) and subsequently reviewed, edited, and verified for technical accuracy by the authors.

---

In [ ]:
# ── Standard library ────────────────────────────────────────────────────────────
import os
import sys
import pickle
import shutil
import random

# ── Progress bars ───────────────────────────────────────────────────────────────
from tqdm import tqdm

# ── Geospatial ──────────────────────────────────────────────────────────────────

import osmnx as ox
import networkx as nx
import geopandas as gpd
import shapely

# ── Data science ────────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Plotting ────────────────────────────────────────────────────────────────────
from matplotlib import pyplot as plt

# ── Display configuration ───────────────────────────────────────────────────────
import warnings; warnings.simplefilter('ignore')
pd.options.display.max_columns = None
pd.options.display.max_rows = 30
np.set_printoptions(threshold=sys.maxsize)

---
## Section 2. Attribute Flood Inundation to Network Nodes and Edges

Each road network graph is loaded and flood water depths are sampled from raster mosaics for all 10 return periods. The process:

1. **Node-level sampling** — For each return period, the function `gn.sample_raster()` queries the corresponding flood raster at each node's geographic coordinates and stores the extracted depth as a node attribute `FUP_<RP>` (metres).

2. **Edge-level attribution** — Because edges represent road segments between two intersections, the edge flood depth is set to the **maximum** of its two endpoint node depths. This conservative assignment ensures that a road segment is considered flooded if either end is inundated, reflecting the physical reality that the highest point of inundation along a segment controls passability.

**Input:** Base road network graphs (`grth_90_graphs_drive_service_no_res/`)  
**Output:** Flood-attributed graphs saved to `grth_90_graphs_10flooded/`

In [ ]:
# Define the 10 flood return periods (years) to be analysed
RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]

for file in tqdm(os.listdir('L:/yiyi/grth_90_graphs_drive_service_no_res/')):

    # Extract the network ID from the filename (characters 15 onwards, strip '.pk' extension)
    net_id = int(file[15:][:-3])

    # Load the base road network graph
    G = pickle.load(open('L:/yiyi/grth_90_graphs_drive_service_no_res/'+ file, 'rb'))
    G_flooded = G.copy()

    # --- Step 1: Sample flood depths at each node ---
    # For each return period, extract the raster flood depth at every node location.
    # The raster mosaic file is named FUP<RP>_mosaic.tif.
    # The sampled depth (metres) is stored as node attribute 'FUP_<RP>'.
    for rp in RP_lst:
        G_flooded = gn.sample_raster(G_flooded, 'J:/yiyi/mosaics/FUP'+ str(rp) +'_mosaic.tif', 'FUP_'+ str(rp))
    
    # --- Step 2: Propagate flood depths from nodes to edges ---
    # An edge connects two nodes (i, j). The edge flood depth is assigned as the
    # maximum of the two endpoint depths. This conservative rule ensures that
    # a segment is treated as flooded if either junction is inundated.
    for i, j, data in G_flooded.edges.data():
        for rp in RP_lst:
            FUP_i = G_flooded.nodes[i]['FUP_'+str(rp)]
            FUP_j = G_flooded.nodes[j]['FUP_'+str(rp)]
            G_flooded[i][j][0]['FUP_'+str(rp)] = max(FUP_i, FUP_j)

    with open('L:/yiyi/grth_90_graphs_10flooded/G_cov_no_r_' + str(net_id) + '_10flooded.pk', 'wb') as handle:
                pickle.dump(G_flooded, handle, protocol=2)

---
## Section 3. Remove Flooded Elements and Adjust Travel Speeds

This cell applies a **0.3 m (300 mm) inundation threshold** to simulate network disruption. Elements exceeding this depth are considered impassable. For remaining (partially flooded) edges, travel speed is reduced using the empirical depth–speed model.

### Processing Pipeline (per graph, per return period)

| Step | Action | Detail |
|---|---|---|
| 1 | **Node removal** | Remove nodes where `FUP_<RP> ≥ 0.3 m` |
| 2 | **Edge removal** | Remove edges where `FUP_<RP> ≥ 0.3 m` |
| 3 | **Dry speed assignment** | Add designed (dry) speeds via `ox.speed.add_edge_speeds()` |
| 4 | **Speed reduction** | Apply depth–speed formula; cap at designed speed |
| 5 | **Travel time update** | Recompute travel times from updated speeds |
| 6 | **Save outputs** | Save disrupted graph per network and return period |

### Speed–Depth Model
The quadratic relationship used to estimate flood-constrained vehicle speed:

$$v_{flood} = 86.9448 - 0.5529 \cdot d + 0.0009 \cdot d^2 \quad [\text{km/h}]$$

where $d$ = water depth in **millimetres**. The effective speed is: $v_{eff} = \min(v_{flood},\, v_{design})$

> **Note:** Failed graphs (e.g., disconnected or empty graphs after node removal) are recorded in `problem_graph_ids` for post-hoc inspection rather than halting the pipeline.

**Input:** Flood-attributed graphs (`2616_flooded_graphs_drive_servce_no_res/`)  
**Output:** Disrupted graphs with updated speeds and travel times (`2616_disprupted_graphs/FUP_<RP>/`)

In [ ]:
# Define return periods and input directory
RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]
flooded_graphs_2616_dir = 'L:/yiyi/2616_flooded_graphs_drive_servce_no_res/'

# Track any graphs that fail processing for diagnostic review
problem_graph_ids = []

# Enter loop
for rp in RP_lst:
    print(rp)
    for flooded_graph_file in os.listdir(flooded_graphs_2616_dir):

        # Extract network ID from filename
        net_id = int(flooded_graph_file[11:][:-13])

        # Load the flood-attributed graph and create a mutable copy for disruption
        G_flooded =  pickle.load(open(flooded_graphs_2616_dir + flooded_graph_file, 'rb'))
        G_disrupted = G_flooded.copy()
        
        # --- Step 1 & 2: Remove inundated nodes and edges (threshold = 0.3 m) ---
        # Nodes and edges with flood depth >= 0.3 m are treated as physically impassable.
        # Node removal is applied to G_disrupted while iterating over G_flooded
        # to avoid modifying the collection during iteration.
        try:
            # Delete nodes with flood inundation greater than 30mm or 0.3m
            for node_id, node_data in G_flooded.nodes.data():
                if node_data['FUP_' + str(rp)] >= 0.3:
                    G_disrupted.remove_node(node_id)
            # Delete edges with flood inundation greater than 30mm or 0.3m
            for start_id, end_id, edge_data in G_flooded.edges.data():
                if edge_data['FUP_' + str(rp)] >= 0.3:
                    G_disrupted.remove_edge(start_id, end_id)
        except:
            problem_graph_ids.append(net_id)

        # --- Step 3: Assign dry (designed) speeds to surviving edges ---
        # osmnx infers speed from OSM highway tags where explicit values are absent.
        try:
            G_disrupted_speed = ox.speed.add_edge_speeds(G_disrupted)
        except:
            problem_graph_ids.append(net_id)
            continue # Skip to next graph; cannot compute travel times without speed

        # --- Step 4 & 5: Apply flood speed reduction and recompute travel times ---
        # For each remaining edge, compute the flood-reduced speed using the
        # empirical quadratic model. The edge speed is capped at the designed speed
        # to prevent physically unrealistic speed increases at very shallow depths.
        try:
            for start_id, end_id, edge_data in G_disrupted_speed.edges.data():
                # Water depth in mm from the edge attribute
                edge_water_depth_mm = edge_data['FUP_' + str(rp)]

                # Quadratic depth–speed relationship (depth in mm, speed in km/h)
                theoretical_speed = 86.9448 - (0.5529*edge_water_depth_mm) + (0.0009*edge_water_depth_mm*edge_water_depth_mm)
                designed_speed = edge_data['speed_kph']

                # Effective speed: flood-reduced but never exceeding the road design speed
                G_disrupted_speed[start_id][end_id][0]['speed_kph'] = min(theoretical_speed, designed_speed)
            
            # Recompute travel times (seconds) from updated edge speeds and lengths
            G_disrupted_speed_time = ox.speed.add_edge_travel_times(G_disrupted_speed)
        except:
             problem_graph_ids.append(net_id)
      # --- Step 6: Save the disrupted graphs ---
        with open('L:/yiyi/2616_disprupted_graphs/FUP_' + str(rp) + '/G_' + str(net_id) + '_disrupted_FUP_' + str(rp) + '.pk', 'wb') as handle:
            pickle.dump(G_disrupted_speed_time, handle, protocol=2)

---
## Section 4. Flood Exposure Analysis

This section quantifies **road network exposure** as a function of both flood return period and inundation depth threshold.

### Method
For each network, each return period, and each depth threshold:
- Identify edges where `FUP_<RP> ≥ threshold`
- Sum their lengths (metres) to compute total exposed road length

### Output Schema
The resulting CSV has the structure:

| Column | Description |
|---|---|
| `net_id` | Network identifier |
| `RP` | Flood return period (years) |
| `0.0` … `5.0` | Total exposed road length (m) at each depth threshold |

**Output:** `../1_processed/exposure_sensitivity_raw_summary.csv`

In [ ]:
# Input directory and return period list
flooded_graph_dir = 'L:/yiyi/all_graph_flooded/'
flood_10_rps = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]

# Initialise results DataFrame; rows are appended per (network, return period) pair
df = pd.DataFrame()

for file in tqdm(os.listdir(flooded_graph_dir)):

    # Process only pickle files (skip any auxiliary files)
    if file[-2:] == 'pk':

        # Extract network ID from filename
        net_id = int(file[11:-13])

        # Load the flood-attributed graph
        G = pickle.load(open('L:/yiyi/all_graph_flooded/'+ file, 'rb'))

        # Build a DataFrame of edge lengths, indexed by (start_node, end_node, key)
        graph_length_df = pd.DataFrame.from_dict(nx.get_edge_attributes(G, "length"), orient='index').rename(columns={0:'length'})
        for rp in flood_10_rps:
            graph_length_df[f'FUP_{rp}_m'] = pd.DataFrame.from_dict(nx.get_edge_attributes(G, f"FUP_{rp}"), orient='index')[0]
        
        # Attach flood depth columns for all return periods
        for rp in flood_10_rps:
            ls = [net_id, rp]
            for threshold in np.arange(0.0, 5.1, 0.1):
                graph_len = graph_length_df[graph_length_df[f'FUP_{rp}_m'] >= threshold]['length'].sum()
                ls.append(graph_len)
                
            df = df.append(pd.DataFrame([ls],
                                        columns=['net_id','RP']+[str(round(i,1)) for i in np.arange(0.0, 5.1, 0.1)]),
                               ignore_index = True)
df.to_csv('../1_processed/exposure_sensitivity_raw_summary.csv')

---
## Section 5. Generate Final Disrupted Networks

This is the production version of the disruption pipeline (cf. Section 3). It applies the same 0.3 m disruption threshold and depth–speed model, with more detailed diagnostic logging and a minor methodological refinement:

### Refinement vs. Section 3
Edge water depth is re-derived from the surviving **node** attributes (not the edge attribute directly):

```python
edge_water_depth_mm = max(
    G_disrupted_speed.nodes[start_id]['FUP_<RP>'],
    G_disrupted_speed.nodes[end_id]['FUP_<RP>']
) * 1000
```

This approach is more robust post-disruption because it uses the current node state rather than the pre-disruption edge attribute, ensuring internal consistency of the graph after node removal.

### Processing Pipeline

| Step | Action |
|---|---|
| 1 | Load flood-attributed graph |
| 2 | Remove nodes with inundation ≥ 0.3 m (edges to/from removed nodes are dropped automatically) |
| 3 | Assign dry designed speeds (`ox.speed.add_edge_speeds`) |
| 4 | Compute flood-reduced speeds from node-derived water depths |
| 5 | Recompute travel times (`ox.speed.add_edge_travel_times`) |
| 6 | Serialise disrupted graph to disk |

> **Note on node-only removal:** In this version, only nodes are explicitly removed. NetworkX automatically removes edges connecting to deleted nodes, so a separate edge-removal step is unnecessary here.

**Input:** Flood-attributed graphs (`2616_flooded_graphs_drive_servce_no_res/`)  
**Output:** Final disrupted graphs (`2616_disrupted_graphs/FUP_<RP>/`)

In [ ]:
# Define return periods and input directory
RP_lst = [5, 10, 20, 50, 75, 100, 200, 250, 500, 1000]
flooded_graphs_2616_dir = 'L:/yiyi/2616_flooded_graphs_drive_servce_no_res/'

for rp in RP_lst:
    print('RP=' + str(rp))
    for flooded_graph_file in os.listdir(flooded_graphs_2616_dir):
        net_id = int(flooded_graph_file[11:][:-13])
        print('net_id='+ str(net_id))

        # Load flood-attributed graph and create mutable copy
        G_flooded =  pickle.load(open(flooded_graphs_2616_dir + flooded_graph_file, 'rb'))
        G_disrupted = G_flooded.copy()

        # --- Step 1: Remove nodes inundated at or above 0.3 m ---
        # Iterating over G_flooded while modifying G_disrupted prevents
        # 'dictionary changed size during iteration' errors.
        # Note: NetworkX automatically removes incident edges when a node is removed.
        for node_id, node_data in G_flooded.nodes.data():
            if node_data['FUP_' + str(rp)] >= 0.3:
                G_disrupted.remove_node(node_id)

        print('removed: '+ str(G_disrupted.number_of_nodes()) + ' from ' + str(G_flooded.number_of_nodes()))

        # --- Step 2: Assign dry designed speeds ---
        # osmnx infers speed from OSM highway tags; required before travel time computation.
        try:
            G_disrupted_speed = ox.speed.add_edge_speeds(G_disrupted)
            print('finished adding ori speed')
        except:
            print('CANNOT add ori speed')
            continue

        # --- Step 3: Apply flood speed reduction ---
        # For each surviving edge, derive water depth from the maximum of the two
        # endpoint node depths (node attributes are more reliable post-disruption
        # than pre-disruption edge attributes).
        for start_id, end_id, edge_data in G_disrupted_speed.edges.data():

            # Derive edge depth as the maximum of the two endpoint node depths
            edge_water_depth_mm = max(G_disrupted_speed.nodes[start_id]['FUP_'+str(rp)], G_disrupted_speed.nodes[end_id]['FUP_'+str(rp)])
            
            # Quadratic speed–depth relationship
            theoretical_speed = 86.9448 - (0.5529*edge_water_depth_mm) + (0.0009*edge_water_depth_mm*edge_water_depth_mm)
            designed_speed = edge_data['speed_kph']

            # Cap flood-reduced speed at the road's designed speed
            G_disrupted_speed[start_id][end_id][0]['speed_kph'] = min(theoretical_speed, designed_speed)
        
        # --- Step 4: Recompute edge travel times ---
        try:
            G_disrupted_speed_time = ox.speed.add_edge_travel_times(G_disrupted_speed)
            print('finished changing speed')
        except:
            print('CANNOT disrupt speed and time')
#         print('L:/yiyi/2616_disrupted_graphs/FUP_' + str(rp) + '/G_' + str(net_id) + '_disrupted_FUP_' + str(rp) + '.pk')
        
        # --- Step 5: Output the final disrupted graph ---
        with open('L:/yiyi/2616_disrupted_graphs/FUP_' + str(rp) + '/G_' + str(net_id) + '_disrupted_FUP_' + str(rp) + '.pk', 'wb') as handle:
            pickle.dump(G_disrupted_speed_time, handle, protocol=2)